#### Import des librairies

In [29]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [30]:

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

#### Import des données

In [31]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'espece', 'culture', 'composant_culture', 'noeuds_realise', 'zone', 'sdc', 'noeuds_synthetise_restructure', 'noeuds_synthetise',
    'connection_synthetise', 'plantation_perenne_phases_realise', 'plantation_perenne_realise',
    
    'sdc', 'domaine'
]

tables_without_id = [
    'typologie_assol_can_realise', 'typologie_can_culture', 'itk_realise_agrege', 'itk_synthetise_agrege', 
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 4/4 [00:04<00:00,  1.07s/it]


In [32]:
left = df['plantation_perenne_phases_realise']
right = df['itk_realise_agrege'].set_index('itk_id')
df['plantation_perenne_phases_realise_extanded'] = pd.merge(left, right, left_index=True, right_index=True, how='left')

left = df['noeuds_realise']
right = df['itk_realise_agrege'].set_index('itk_id')[['sdc_id']]
df['noeuds_realise_extanded'] = pd.merge(left, right, left_index=True, right_index=True, how='left')

In [33]:
studied_ids = [
    # Porte graine
    'fr.inra.agrosyst.api.entities.GrowingSystem_88bd9f2a-5604-4171-9b00-26af988f5441',
    'fr.inra.agrosyst.api.entities.GrowingSystem_ac66c1bc-1a96-492b-bf76-64087319872d',
    'fr.inra.agrosyst.api.entities.GrowingSystem_c7d5fc9e-4560-4a2c-8b1e-7af04779b820',
    #Divers assolé
    'fr.inra.agrosyst.api.entities.GrowingSystem_8f9c8256-bb5f-46f8-89bd-9c1d89a5e4c1',
    'fr.inra.agrosyst.api.entities.GrowingSystem_219d626f-9be8-431a-9458-d491820d1dc8',
    'fr.inra.agrosyst.api.entities.GrowingSystem_f4466535-63ba-4afd-a63e-a1a4afad4965',
    'fr.inra.agrosyst.api.entities.GrowingSystem_97dd5d5c-e75c-49c2-bdc2-dcddf4c53244',
    'fr.inra.agrosyst.api.entities.GrowingSystem_240b8c12-8982-47be-8313-5bcccfc6ec22',
    'fr.inra.agrosyst.api.entities.GrowingSystem_8b2b677f-572e-4f04-abbf-ca56da468416',
    'fr.inra.agrosyst.api.entities.GrowingSystem_751c6351-04cd-44e3-94f1-7d84c66b1622', 

    # Divers Perenne
    'fr.inra.agrosyst.api.entities.GrowingSystem_02e4f6e5-fe8b-4aae-9892-1e25b9d3fa4a',
    'fr.inra.agrosyst.api.entities.GrowingSystem_2708a1f9-737f-442d-ace7-34e432c7ea04',
    'fr.inra.agrosyst.api.entities.GrowingSystem_1cb96b86-e8a4-44a2-b9a8-e1cc45fe3aa4',
    'fr.inra.agrosyst.api.entities.GrowingSystem_614db553-f89c-45d6-bd68-7561c47a95dc'
]

In [34]:
df['itk_realise_agrege_test'] = df['itk_realise_agrege'].loc[
    df['itk_realise_agrege']['sdc_id'].isin(studied_ids)
]

In [35]:
df['noeuds_realise_test']= df['noeuds_realise'].loc[
    df['noeuds_realise'].index.isin(df['itk_realise_agrege_test']['itk_id'])
]

df['plantation_perenne_phases_realise_test'] = df['plantation_perenne_phases_realise'].loc[
    df['plantation_perenne_phases_realise'].index.isin(df['itk_realise_agrege_test']['itk_id'])
]

df['plantation_perenne_realise_test'] = df['plantation_perenne_realise'].loc[
    df['plantation_perenne_realise'].index.isin(df['itk_realise_agrege_test']['plantation_perenne_realise_id'])
]

df['noeuds_realise_test']= df['noeuds_realise'].loc[
    df['noeuds_realise'].index.isin(df['noeuds_realise_test'].index)
]
df['itk_realise_agrege_test'] = df['itk_realise_agrege'].loc[
    df['itk_realise_agrege']['itk_id'].isin(df['noeuds_realise_test'].index) |
    df['itk_realise_agrege']['itk_id'].isin(df['plantation_perenne_phases_realise_test'].index)
]
df['zone_test'] = df['zone'].loc[
    df['zone'].index.isin(df['itk_realise_agrege_test']['zone_id'])
]
df['sdc_test'] = df['sdc'].loc[
    df['sdc'].index.isin(df['itk_realise_agrege_test']['sdc_id'])
]
df['domaine_test'] = df['domaine'].loc[
    df['domaine'].index.isin(df['itk_realise_agrege_test']['domaine_id'])
]
df['typologie_can_culture_test'] = df['typologie_can_culture'].loc[
    df['typologie_can_culture']['culture_id'].isin(df['noeuds_realise_test']['culture_id']) |
    df['typologie_can_culture']['culture_id'].isin(df['plantation_perenne_realise_test']['culture_id'])
]

In [36]:
df['plantation_perenne_phases_realise_test']

,duree,type,plantation_perenne_realise_id
id,,,
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_8ee8481e-97c8-49f0-90dd-67599475fb58,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_e51b51dd-063a-4f20-b3ec-325765094541,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_6e7da619-2b76-474a-b1f5-9e6416f8fa31,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_56927355-b786-413c-b2b7-5fee744d8d60,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_e92be9c8-1df1-4aee-a164-1a4a07e85911,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_50d5e5b8-cd46-485c-b174-39255ca2deab,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_a28d6543-da47-4740-87b8-3e44296d408f,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_b57b67ff-f618-4625-bb9c-0f35acf5073b,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...
fr.inra.agrosyst.api.entities.effective.EffectiveCropCyclePhase_c8b62cb3-7560-4c2f-b065-08e6152a74a7,NaN,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.effective.Effect...


In [37]:
# export 
path='./'
df['noeuds_realise_test'].to_csv(path+'noeuds_realise.csv')
df['itk_realise_agrege_test'].to_csv(path+'itk_realise_agrege.csv')
df['zone_test'].to_csv(path+'zone'+'.csv')
df['typologie_can_culture_test'].to_csv(path+'typologie_can_culture'+'.csv')
df['domaine_test'].to_csv(path+'domaine'+'.csv')
df['sdc_test'].to_csv(path+'sdc'+'.csv')
df['plantation_perenne_phases_realise_test'].to_csv(path+'plantation_perenne_phases_realise'+'.csv')
df['plantation_perenne_realise_test'].to_csv(path+'plantation_perenne_realise'+'.csv')